# Study 894 — Trend Overlay on 60/40 📉

**Does a 200-day trend filter cut the balanced book's drawdown *and* keep its return?**

Take the classic **60% SPY / 40% IEF** book and lay Faber's 200-day moving-average filter
over *each leg*: hold the equity sleeve while SPY is above its 200-day MA, the bond sleeve
while IEF is above its own — step whichever has rolled over to **BIL** cash. The pitch is a
free lunch: keep most of the 60/40's return while dodging its worst drawdowns
(2007-05-30 → 2026-06-30, 4,602 days, excess of cash).

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `0dd2af7e1636`);
the live cell runs the fast synthetic control. Short history: BIL (cash) launches 2007, so
this is a one-crash-each sample — named on the Signal axis.*


## 1. The idea in one picture

A 200-day moving average is a slow trend gauge: above it, the asset is in an uptrend; below it, a downtrend. Hold each leg only while it is *above* its own MA and park it in T-bills otherwise, and the book should step out of the way of long bear markets — 2008 for stocks, 2022 for bonds — the two episodes the static 60/40 is built to survive and doesn't.

In [1]:
R = dict(sharpe_strat=0.823, sharpe_bench=0.689, sharpe_adv=0.134,
         maxdd_strat=-12.5, maxdd_bench=-30.8, dd_cut=18.3,
         vol_strat=6.9, vol_bench=11.4, cagr_strat=6.94, cagr_bench=8.87,
         diff_bps=-0.87, t_nw=-1.18)
print('DRAWDOWN : overlay %+.1f%%  vs  static %+.1f%%   (cut %+.1f pp)'
      % (R['maxdd_strat'], R['maxdd_bench'], R['dd_cut']))
print('VOL      : overlay %.1f%%   vs  static %.1f%%' % (R['vol_strat'], R['vol_bench']))
print('SHARPE   : overlay %.2f   vs  static %.2f   (adv %+.2f)'
      % (R['sharpe_strat'], R['sharpe_bench'], R['sharpe_adv']))
print('CAGR     : overlay %.2f%%  vs  static %.2f%%   <- gives up return for the calm'
      % (R['cagr_strat'], R['cagr_bench']))

DRAWDOWN : overlay -12.5%  vs  static -30.8%   (cut +18.3 pp)
VOL      : overlay 6.9%   vs  static 11.4%
SHARPE   : overlay 0.82   vs  static 0.69   (adv +0.13)
CAGR     : overlay 6.94%  vs  static 8.87%   <- gives up return for the calm


## 2. Where the edge lives — the two crash years

In **2008** the overlay returned **+2.1%** while the static book lost **-13.2%**; in **2022** (stocks *and* bonds down together) **-8.5%** vs **-16.4%**. In the calm years in between it *lags*, dripping away return on false signals and sitting in low-yield cash. It is tail insurance, not a return engine.

## 3. Is the sort just lucky? A live synthetic control

Plant deep, persistent bear regimes the 200-day filter can duck (`edge=1`), and a flat null with no bear to duck (`edge=0`). The overlay must recover a drawdown cut and a Sharpe pickup in the planted world, and do neither on the null. Live, no network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from trend6040 import data, strategy as st
# average a few seeds so the demo is representative, not one lucky/unlucky world
plant = [st.synthetic_detect(data.synthetic_prices(edge=1.0, seed=894+s, n_days=5000)) for s in range(6)]
null  = [st.synthetic_detect(data.synthetic_prices(edge=0.0, seed=894+s, n_days=5000)) for s in range(6)]
pdd, padv = np.mean([d['dd_cut'] for d in plant])*100, np.mean([d['sharpe_adv'] for d in plant])
ndd, nadv = np.mean([d['dd_cut'] for d in null])*100, np.mean([d['sharpe_adv'] for d in null])
print('planted world (6 seeds): DD cut %+.1f pp, Sharpe adv %+.2f  (both light up)' % (pdd, padv))
print('null world    (6 seeds): DD cut %+.1f pp, Sharpe adv %+.2f  (~ no skill)'   % (ndd, nadv))

planted world (6 seeds): DD cut +38.8 pp, Sharpe adv +0.27  (both light up)
null world    (6 seeds): DD cut +20.3 pp, Sharpe adv +0.01  (~ no skill)


## 4. The honest verdict — real risk cut, no bankable free lunch

On the real tape the overlay **roughly halves the drawdown** (-12.5% vs -30.8%) and the volatility — a genuine, mechanical benefit. But it *keeps the return* it claims to only in a loose sense: it **gives up ~1.9 pp/yr of CAGR**, the daily return difference is negative and insignificant (-0.87 bps/day, NW *t* = -1.18), and the Sharpe advantage of +0.13 has a bootstrap CI that **straddles zero** ([-0.24, +0.52], positive in only 74% of resamples). Worse, a **25% short-term tax** on each forced move to cash flips that thin edge to -0.28. **Signal: Weak** (real drawdown cut, no robust Sharpe edge), **Tradability: Fragile** (survives trading costs, dies to tax and time).